# F5-probability — Session 03: The Gaussian, Simulation, and Covariance

**Session length:** about 85 minutes • **Concepts:** gaussian-distribution,
sampling-simulation, covariance — plus the unit's exam connections and where
these ideas go next.

Checkpoint answers are collected at the end of the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Sums pile up in a bell

Session 02 computed the expectation and variance of a sum of many
independent variables. Now look at the sum's whole *shape*. One die is flat —
every face equally likely. But watch histograms of the total of 2, then 5,
then 50 dice:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
for ax, n_dice in zip(axes, [1, 2, 5, 50]):
    totals = rng.integers(1, 7, size=(50_000, n_dice)).sum(axis=1)
    ax.hist(totals, bins=min(30, totals.max() - totals.min() + 1))
    ax.set_title(f"total of {n_dice} dice")
    ax.set_xlabel("total")
axes[0].set_ylabel("how many runs")
plt.tight_layout()
plt.show()

Flat → triangle → mound → smooth bell. The same bell emerges from coins,
spinners, *any* independent same-shaped contributions — by 50 terms the
original flatness of a die is unrecognizable. This is a deep (and here,
stated) fact of probability: **sums of many independent contributions are
bell-shaped**, almost regardless of what is being summed. Nature adds up
many small independent effects constantly, which is why measurement noise,
heights, and test-score totals keep showing this same curve.

### Checkpoint 1

1. Rerun the experiment summing 50 *spinner* draws
   (values $1, 2, 5$ with $p = 0.5, 0.25, 0.25$) instead of dice. Is the
   bell there too, even though one spinner's histogram is wildly lopsided?
2. Using Session 02's laws: compute $E$ and $\operatorname{Var}$ of the
   50-dice total, and check both against the simulated `totals` array
   ($E[X] = 3.5$, $\operatorname{Var}[X] = 35/12$ per die).

## 2. The Gaussian, stated

The limiting bell shape has a name — the **Gaussian** (or *normal*)
distribution with parameters $\mu$ and $\sigma^2$, written
$\mathcal{N}(\mu, \sigma^2)$. Its bell curve is the graph of

$$f(x) \;=\; \frac{1}{\sigma\sqrt{2\pi}}\;
 e^{-\frac{(x - \mu)^2}{2\sigma^2}},$$

centered at $\mu$, with width set by $\sigma$. A Gaussian random variable
is *continuous* — it can take any decimal value, not just table entries, so
probabilities are **areas under the curve** rather than sums of table rows.

**How this course handles the continuous case, stated once and used
throughout:** these areas are not computable by Calculus AB
antiderivatives — the function $e^{-x^2}$ famously has no elementary
antiderivative. So Gaussian facts enter this course as *stated facts*, and
every one of them is **verified by seeded simulation** instead of by
integration. The facts to know:

1. If $X \sim \mathcal{N}(\mu, \sigma^2)$ then $E[X] = \mu$ and
   $\operatorname{Var}[X] = \sigma^2$ — the parameters *are* the
   expectation and variance.
2. $\mathcal{N}(0, 1)$ is called the **standard** Gaussian.
3. The **68–95–99.7 rule**: about $68.3\%$ of draws land within
   $1\sigma$ of $\mu$, $95.4\%$ within $2\sigma$, $99.7\%$ within
   $3\sigma$.

Watch fact 3 emerge from a histogram of the 50-dice totals with the Gaussian
curve overlaid ($\mu = 175$, $\sigma^2 = 1750/12$ from Checkpoint 1):

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
totals = rng.integers(1, 7, size=(50_000, 50)).sum(axis=1)

mu, sigma = 175.0, np.sqrt(1750 / 12)
xs = np.linspace(mu - 4 * sigma, mu + 4 * sigma, 300)
curve = np.exp(-((xs - mu) ** 2) / (2 * sigma ** 2)) / (sigma * np.sqrt(2 * np.pi))

plt.figure(figsize=(7, 4))
plt.hist(totals, bins=40, density=True, label="50-dice totals (simulated)")
plt.plot(xs, curve, linewidth=2, label="Gaussian curve N(175, 1750/12)")
plt.title("A sum of independent contributions vs. the Gaussian")
plt.xlabel("total")
plt.ylabel("density")
plt.legend()
plt.show()

print("within 1 sigma:", (np.abs(totals - mu) <= sigma).mean(), "  (stated: ≈ 0.683)")
print("within 2 sigma:", (np.abs(totals - mu) <= 2 * sigma).mean(), "  (stated: ≈ 0.954)")

(`density=True` rescales the bars so their areas total 1, putting the
histogram and the curve on the same vertical scale.)

### Checkpoint 2

1. Why can't the area under $f(x)$ between $174$ and $176$ be computed the
   Calc AB way, and what does this course use instead?
2. From the 68–95–99.7 rule alone: roughly what fraction of draws from
   $\mathcal{N}(100, 25)$ land in $[95, 105]$? Outside $[90, 110]$?

## 3. Drawing Gaussians and standardizing

NumPy draws Gaussian samples directly: `rng.normal(mu, sigma, size)` — note
it takes $\sigma$, **not** $\sigma^2$. That gives us the simulation tool to
verify all the stated facts wholesale:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
g = rng.normal(100.0, 5.0, size=500_000)          # N(100, 25): sigma = 5

print("sample mean:", g.mean(), "        (fact 1: mu = 100)")
print("sample var: ", ((g - g.mean()) ** 2).mean(), "  (fact 1: sigma^2 = 25)")
print("within 1 sigma:", (np.abs(g - 100) <= 5).mean())
print("within 2 sigma:", (np.abs(g - 100) <= 10).mean())
print("within 3 sigma:", (np.abs(g - 100) <= 15).mean())

**Standardization.** Any Gaussian is a shifted, scaled copy of the standard
one. The recipe from data to standard units:

$$z \;=\; \frac{x - \mu}{\sigma}$$

$z$ counts *how many standard deviations $x$ sits above its expectation* —
a pure number, comparable across totally different measurements. By Session
02's laws, if $X$ has expectation $\mu$ and variance $\sigma^2$ then
$Z = (X - \mu)/\sigma$ has expectation 0 and variance 1 (shift kills
$\mu$, scale by $1/\sigma$ divides the variance by $\sigma^2$) — that
part is *proved*, for any variable. That $Z$ is exactly standard-Gaussian
when $X$ is Gaussian is stated fact territory, verified below.

Standardization turns the 68–95–99.7 rule into a universal yardstick:
$|z| > 2$ is rare (under 5%), $|z| > 3$ is exceptional (0.3%).

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.normal(100.0, 5.0, size=500_000)
z = (x - 100.0) / 5.0                             # standardize

print("z sample mean:", z.mean(), "  z sample var:", ((z - z.mean()) ** 2).mean())

std = rng.normal(0.0, 1.0, size=500_000)          # direct standard draws
plt.figure(figsize=(7, 4))
plt.hist(z, bins=60, density=True, alpha=0.6, label="standardized N(100,25) draws")
plt.hist(std, bins=60, density=True, alpha=0.6, label="direct N(0,1) draws")
plt.title("Standardizing recovers the standard Gaussian")
plt.xlabel("z")
plt.ylabel("density")
plt.legend()
plt.show()

### Checkpoint 3

1. Exam scores are $\mathcal{N}(72, 36)$. Standardize a score of 87. Is it
   more or less impressive than a 79 on a $\mathcal{N}(70, 9)$ exam?
   (Standardize both.)
2. `rng.normal(50, 16, size=...)` — a classmate says this draws from
   $\mathcal{N}(50, 16)$. What distribution does it actually draw from, and
   how would you check their claim with one variance estimate?

## 4. Gaussian arithmetic (stated facts, verified)

Two more stated facts complete the toolkit. Both are "the Gaussian family is
closed under the arithmetic of Sessions 01–02":

4. **Shift/scale:** if $X \sim \mathcal{N}(\mu, \sigma^2)$ and $a, b$
   are constants, then $aX + b$ is Gaussian too — necessarily
   $\mathcal{N}(a\mu + b,\ a^2\sigma^2)$, since expectation and variance
   follow the *proved* laws; the stated part is that the result is still
   Gaussian-shaped.
5. **Sums:** if $X \sim \mathcal{N}(\mu_1, \sigma_1^2)$ and
   $Y \sim \mathcal{N}(\mu_2, \sigma_2^2)$ are independent, then $X + Y$
   is Gaussian — necessarily
   $\mathcal{N}(\mu_1 + \mu_2,\ \sigma_1^2 + \sigma_2^2)$ by
   linearity and the independent-sum law.

Notice the division of labor: *which* $\mu$ and $\sigma^2$ come out is
Session 01–02 algebra, proved; *that the bell shape survives* is the stated
continuous fact. Verification of fact 5:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.normal(3.0, 2.0, size=400_000)            # N(3, 4)
y = rng.normal(-1.0, 1.5, size=400_000)           # N(-1, 2.25), independent draws
s = x + y                                         # claim: N(2, 6.25)

print("sample mean:", s.mean(), "   (claim: 2)")
print("sample var: ", ((s - s.mean()) ** 2).mean(), "   (claim: 6.25)")

ref = rng.normal(2.0, 2.5, size=400_000)          # direct N(2, 6.25) draws
plt.figure(figsize=(7, 4))
plt.hist(s, bins=60, density=True, alpha=0.6, label="X + Y (simulated)")
plt.hist(ref, bins=60, density=True, alpha=0.6, label="direct N(2, 6.25)")
plt.title("Sum of independent Gaussians is Gaussian")
plt.xlabel("value")
plt.ylabel("density")
plt.legend()
plt.show()

### Checkpoint 4

1. $X \sim \mathcal{N}(10, 4)$. Give the exact distribution of
   $-2X + 5$, separating which parts of your answer are proved laws and
   which part is the stated fact.
2. Daily outputs of two independent machines are
   $\mathcal{N}(40, 9)$ and $\mathcal{N}(60, 16)$. Give the distribution
   of the combined daily output and, via 68–95–99.7, an interval that
   contains about 95% of days.

## 5. Monte Carlo: simulation as a measuring instrument

We have been using the same three-step method all unit; name it now.
**Monte Carlo estimation**: to find a probability or an expectation,

1. simulate the process $n$ times (seeded, loop-free);
2. probabilities → the `.mean()` of an event mask;
   expectations → the `.mean()` of the per-run values;
3. trust the estimate more as $n$ grows.

It works because of the frequency-settling behavior observed since Session
01 — and it is the *only* general tool you have when algebra runs out (as it
just did for Gaussian areas). What does an estimate cost in accuracy?
Measure it: estimate $P(Z > 1)$ for standard $Z$ at several $n$, with several
seeds each, and watch the spread of the estimates shrink:

In [ ]:
true_p = 0.15866                                   # stated reference value, P(Z > 1)
for n in (100, 10_000, 1_000_000):
    ests = np.array([(np.random.default_rng(20260804 + k).normal(size=n) > 1).mean()
                     for k in range(5)])
    print(f"n = {n:>9,}:  estimates {np.round(ests, 4)}   max error {np.abs(ests - true_p).max():.4f}")

Each 100-fold increase in $n$ buys roughly one more
reliable decimal digit — the error shrinks like $1/\sqrt{n}$ (an observed
pattern here; Session 02's Checkpoint 8 already proved the variance side of
it: averaging $n$ independent repeats divides variance by $n$, hence
$\sigma$ by $\sqrt{n}$).

Practical rules for exam-grade Monte Carlo:

- **Seed it** — an unseeded estimate is unverifiable.
- **Size it** — $n \ge 100{,}000$ for two stable decimals; more for tail
  probabilities.
- **Mask it** — the event is a boolean expression; its `.mean()` is the
  estimate. No loops.

### Checkpoint 5

1. Estimate $P(|Z| > 2.5)$ for standard Gaussian $Z$ with $n = 10^6$
   (seeded). Why would $n = 1{,}000$ be reckless for this particular event?
2. Estimate $E[\max(Z, 0)]$ for standard $Z$ (the average of the positive
   part) — which F1 function makes the per-run values loop-free?

## 6. Covariance

Variance treats one variable alone; **covariance** measures how two move
*together*. Definition and shortcut (proof of the shortcut mirrors Session
02 Section 2 exactly — expand and apply linearity):

$$\operatorname{Cov}[X, Y] = E\big[(X - E[X])(Y - E[Y])\big]
 = E[XY] - E[X]E[Y].$$

Reading the sign: when $X$ above its expectation tends to come with $Y$
above its expectation, the products of deviations are mostly positive →
positive covariance; opposite tendencies → negative; no tendency → near 0.
Three immediate consequences:

- $\operatorname{Cov}[X, X] = \operatorname{Var}[X]$ — covariance
  generalizes variance.
- If $X \perp Y$: $\operatorname{Cov}[X, Y] = E[XY] - E[X]E[Y] = 0$, by
  Session 02's product rule.
- The **general** sum law, no independence needed (expand
  $E[((X{-}\mu_X) + (Y{-}\mu_Y))^2]$):
  $$\operatorname{Var}[X + Y] = \operatorname{Var}[X] +
    \operatorname{Var}[Y] + 2\operatorname{Cov}[X, Y].$$
  Session 02's independent case is this with the covariance term dead.

From $n$ paired samples $(x_i, y_i)$, the estimate is the mean of deviation
products — indexed sums again:
$\widehat{\operatorname{Cov}} = \frac1n \sum_i (x_i - \bar x)(y_i - \bar y)$:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
n = 200_000

study = rng.normal(5.0, 2.0, size=n)                        # hours studied
score = 50.0 + 6.0 * study + rng.normal(0.0, 4.0, size=n)   # tends to rise with study
sleep_loss = -0.5 * study + rng.normal(0.0, 1.0, size=n)    # tends to fall
shoe = rng.normal(26.0, 1.5, size=n)                        # unrelated

def est_cov(a, b):
    return ((a - a.mean()) * (b - b.mean())).mean()

print("Cov[study, score]:", est_cov(study, score), "  (positive: rise together)")
print("Cov[study, sleep_loss]:", est_cov(study, sleep_loss), "  (negative)")
print("Cov[study, shoe]: ", est_cov(study, shoe), "  (≈ 0: unrelated)")
print()
print("general sum law, dependent pair X = study, Y = score:")
lhs = est_cov(study + score, study + score)                 # Var via Cov[S, S]
print("  Var[X + Y]:", lhs)
print("  Var[X] + Var[Y] + 2 Cov:",
      est_cov(study, study) + est_cov(score, score) + 2 * est_cov(study, score))

**The one-way street.** Independence forces covariance 0, but covariance 0
does **not** force independence. Covariance detects only *straight-line*
co-movement; a perfectly dependent but symmetric relationship can hide from
it. The classic counterexample, fully discrete: $X$ uniform on
$\{-1, 0, 1\}$ and $Y = X^2$.

- $E[X] = 0$, and $E[XY] = E[X^3] = \tfrac13(-1 + 0 + 1) = 0$, so
  $\operatorname{Cov} = E[XY] - E[X]E[Y] = 0 - 0 = 0$.
- Yet $Y$ is a *function* of $X$ — maximal dependence:
  $P(X{=}1, Y{=}0) = 0$ while $P(X{=}1)P(Y{=}0) = \tfrac13 \cdot
  \tfrac13 = \tfrac19 \ne 0$.

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.choice(np.array([-1.0, 0.0, 1.0]), size=300_000)
y = x ** 2

print("Cov[X, Y] est:", ((x - x.mean()) * (y - y.mean())).mean(), " ≈ 0")
print("but P(X=1 and Y=0):", ((x == 1) & (y == 0)).mean(),
      " vs P(X=1)P(Y=0):", (x == 1).mean() * (y == 0).mean())

### Checkpoint 6

1. By hand, from the joint outcomes: one fair coin flip $X$ (0 or 1) and
   $Y = 10X$. Compute $E[X]$, $E[Y]$, $E[XY]$, and
   $\operatorname{Cov}[X, Y]$ via the shortcut.
2. Two variables have $\operatorname{Var}[X] = 4$,
   $\operatorname{Var}[Y] = 9$, $\operatorname{Cov}[X, Y] = -3$. Compute
   $\operatorname{Var}[X + Y]$. Is the sum steadier or wilder than
   independence would predict, and why does that make sense?

## 7. Exam connections

How this unit's concepts show up in Round 1 (paraphrased shapes, no verbatim
past-test text; see the probability-statistics cluster in
`reference/analysis.md`):

- **The variance-propagation derivation.** The signature probability
  sub-part: scaled sums of independent factors, exactly Session 02 Section
  7 — *given independent zero-expectation factors, choose the scale factor
  so the sum's variance stays fixed*. It is scored as reasoning
  ("Reasoning is required"), so the derivation itself — cross terms dying by
  independence — is the answer, not just the final constant. Later parts of
  such an arc consume the derived constant explicitly.
- **Seeded-simulation tasks.** Coding sub-parts fix a seed and demand exact
  contracts (function names, shapes) with API bans and a zero-points
  clause — Monte Carlo estimates, empirical frequencies, variance computed
  from raw deviations rather than a library call.
- **Concept multiple choice.** Five options A–E in the opening block:
  which statements about expectation/variance/independence are true, spot
  the broken distribution table, pick the variance of a transformed
  variable. Numeric MC answers may demand a normal form (fraction in lowest
  terms, positivity conditions) that makes the answer unique.

**Worked exam-style example (numeric normal form).** In the exam's register:

> $X$ and $Y$ are independent, $\operatorname{Var}[X] = \tfrac12$,
> $\operatorname{Var}[Y] = \tfrac13$. Write
> $\operatorname{Var}[3X - 2Y]$ as a fraction $p/q$ in lowest terms with
> $q > 0$, and report $S = p + q$.
>
> (A) 6  (B) 41  (C) 47  (D) 65  (E) 71

1. Scale law on each term: $\operatorname{Var}[3X] = 9 \cdot \tfrac12 =
   \tfrac92$; $\operatorname{Var}[-2Y] = 4 \cdot \tfrac13 = \tfrac43$.
2. Independent sum law: $\operatorname{Var}[3X - 2Y] = \tfrac92 +
   \tfrac43 = \tfrac{27 + 8}{6} = \tfrac{35}{6}$.
3. Already in lowest terms ($\gcd(35, 6) = 1$, $q > 0$):
   $S = 35 + 6 = 41$. **Answer (B).**

The normal form is doing real work: an unreduced $\tfrac{70}{12}$ would
give $82$ — not an option, which is your signal to reduce. Simulation
check:

In [ ]:
SEED = 20260804
rng = np.random.default_rng(SEED)
x = rng.normal(0.0, np.sqrt(1 / 2), size=400_000)
y = rng.normal(0.0, np.sqrt(1 / 3), size=400_000)
s = 3 * x - 2 * y
print("Var[3X - 2Y] est:", ((s - s.mean()) ** 2).mean(), "   35/6 =", 35 / 6)

### Checkpoint 7

1. Exam-style, by hand: independent $X, Y$ with
   $\operatorname{Var}[X] = \tfrac14$, $\operatorname{Var}[Y] = 1$.
   $\operatorname{Var}[2X + Y]$ as $p/q$ in lowest terms, $q > 0$; report
   $p + q$: (A) 2 (B) 3 (C) 4 (D) 5 (E) 6. Show steps.
2. Why does a grader's normal-form condition ($\gcd(p, q) = 1$, $q > 0$)
   make a numeric answer *unique*, and what classic error does it catch?

## 8. Common pitfalls III

**Pitfall — trusting small samples.** The unit's simulations use $n$ in the
hundreds of thousands for a reason. Small-sample intuitions are
systematically biased: streaks look meaningful, spread looks smaller than it
is, and rare events look impossible (or, once seen, inevitable). Ten flips
showing 7 heads is *unremarkable* ($P > 0.17$); people read it as a loaded
coin. Ten samples estimating a variance can miss by half:

In [ ]:
samples = np.stack([np.random.default_rng(20260804 + k).normal(0.0, 1.0, size=10)
                    for k in range(8)])                       # 8 tiny studies
ests = ((samples - samples.mean(axis=1, keepdims=True)) ** 2).mean(axis=1)
print("eight 10-sample variance estimates of Var = 1:")
print(np.round(ests, 3))

Each independent run "measures" $\operatorname{Var} = 1$ and reports
anywhere from far below to far above it. The fix is discipline, not
intuition: state $n$, seed it, and distrust any conclusion an $n$ this small
suggests.

**Pitfall — a run of heads makes tails "due".** Independent flips have no
memory: $P(\text{heads}) = \tfrac12$ after any history. The long-run
frequency settles by *swamping* early imbalances with new data, not by
compensating for them.

**Pitfall — covariance 0 read as "unrelated".** Section 6's $Y = X^2$
counterexample: complete dependence, covariance exactly 0. Covariance
speaks only about straight-line co-movement; only the *converse* direction
(independent ⇒ covariance 0) is a theorem.

**Pitfall — 68–95–99.7 applied to non-Gaussians.** The rule is a Gaussian
fact. A single die has about 66% of its mass within $1\sigma$ — close-ish —
but only ~97% within $2\sigma$, and *100%* within $3\sigma$; a heavily
lopsided variable can be far worse. Check the shape (or sum many
contributions) before quoting the rule.

### Checkpoint 8

1. A classmate flips 10 coins, sees 8 heads, and concludes the coin is
   biased; another sees the same and bets on tails next, "because it's
   due". Diagnose both errors in one sentence each.
2. State the Section 6 counterexample (covariance 0 despite complete
   dependence) from memory, with the one-sentence reason covariance misses
   the dependence.

## 9. Going deeper

Nothing below is needed for this unit's practice set — it is the forward
map along the course DAG:

- **C9-dimensionality-reduction** picks up covariance where Section 6
  stopped: with many measurements per observation, the covariances of every
  pair of coordinates, taken together, describe the directions along which
  data genuinely varies — the springboard for compressing
  high-coordinate-count data.
- **C5-neural-networks** runs Session 02 Section 7 in production: the
  scaled-sums identity $\operatorname{Var}[\sum_i w_i x_i] = \sigma^2
  \sum_i w_i^2$ is precisely the tool used there to choose scale factors
  that keep variance stable through repeated summing stages — the same
  choose-$c$-for-target-variance derivation you can already do.
- **C4-data-preprocessing** applies standardization ($z = (x - \mu)/\sigma$,
  Section 3) to real datasets, where putting different measurements on the
  standard scale is a required preparation step.

### Checkpoint 9

1. Warm up for the C5 pattern without leaving this unit: for $n = 400$
   independent zero-expectation factors of variance $2$, what common scale
   factor $c$ keeps the scaled sum at variance 1?
2. Warm up for C9: for the `study`/`score`/`shoe` variables of Section 6,
   which *pair* would a compression method be right to treat as carrying
   overlapping information, and which covariance computation says so?

## Checkpoint Answers

<details><summary><b>Checkpoint 1</b></summary>

1. Yes — the bell appears anyway:

   ```python
   rng = np.random.default_rng(SEED)
   spins = rng.choice(np.array([1.0, 2.0, 5.0]), size=(50_000, 50),
                      p=np.array([0.5, 0.25, 0.25])).sum(axis=1)
   plt.hist(spins, bins=40)
   ```

   The histogram is a clean bell centered near $50 \times 2.25 = 112.5$,
   even though a single spinner's histogram is wildly lopsided — the
   sums-pile-up fact does not care what shape is being summed.

2. Linearity: $E = 50 \cdot 3.5 = 175$. Independent-sum law:
   $\operatorname{Var} = 50 \cdot \tfrac{35}{12} = \tfrac{1750}{12}
   \approx 145.83$. A fresh seeded run of the 50-dice totals gives
   `totals.mean()` → 174.965 and a variance estimate of 144.15 — both
   within a fraction of a percent of the exact values.

</details>

<details><summary><b>Checkpoint 2</b></summary>

1. The area is an integral of a function built on $e^{-x^2}$, which has no
   elementary antiderivative — the Calc AB antidifferentiate-and-evaluate
   route cannot even start. This course estimates such areas by seeded
   Monte Carlo simulation instead: simulate many draws, take the mask mean.

2. $\mathcal{N}(100, 25)$ has $\sigma = 5$. $[95, 105]$ is
   $\mu \pm 1\sigma$: about $68\%$. Outside $[90, 110]$ is outside
   $\mu \pm 2\sigma$: about $100 - 95.4 \approx 4.6\%$.

</details>

<details><summary><b>Checkpoint 3</b></summary>

1. $z = (87 - 72)/6 = 2.5$ versus $z = (79 - 70)/3 = 3.0$ — the 79 is
   *more* impressive: it sits three standard deviations above its exam's
   expectation, the 87 only two and a half.

2. `rng.normal` takes $\sigma$, not $\sigma^2$, so it draws from
   $\mathcal{N}(50, 256)$. One variance estimate settles the claim:

   ```python
   d = np.random.default_rng(SEED).normal(50, 16, size=200_000)
   print(((d - d.mean()) ** 2).mean())   # 256.06 — that is 16², not 16
   ```

</details>

<details><summary><b>Checkpoint 4</b></summary>

1. $-2X + 5 \sim \mathcal{N}(-2 \cdot 10 + 5,\ (-2)^2 \cdot 4)
   = \mathcal{N}(-15, 16)$. Proved parts: the expectation (linearity) and
   the variance (scale law). Stated part: that the result is still
   Gaussian-shaped.

2. Combined output $\sim \mathcal{N}(40 + 60,\ 9 + 16) =
   \mathcal{N}(100, 25)$, so $\sigma = 5$ and about 95% of days fall in
   $[90, 110]$ ($\mu \pm 2\sigma$).

</details>

<details><summary><b>Checkpoint 5</b></summary>

1. ```python
   z = np.random.default_rng(SEED).normal(size=1_000_000)
   print((np.abs(z) > 2.5).mean())   # 0.012428
   ```

   With $n = 1{,}000$ you would expect only about 12 qualifying draws —
   the estimate would swing wildly from seed to seed. Tail probabilities
   need large $n$.

2. `np.maximum(z, 0)` computes every run's positive part loop-free
   (elementwise max against 0 — the F1 function). Its mean on the seeded
   million draws is 0.39992.

</details>

<details><summary><b>Checkpoint 6</b></summary>

1. $E[X] = \tfrac12$; $Y = 10X$ so $E[Y] = 5$. Since $X$ is 0/1,
   $X^2 = X$, so $XY = 10X^2 = 10X$ and $E[XY] = 10 \cdot \tfrac12 = 5$.
   Shortcut: $\operatorname{Cov}[X, Y] = E[XY] - E[X]E[Y]
   = 5 - \tfrac12 \cdot 5 = \tfrac52$.

2. $\operatorname{Var}[X + Y] = 4 + 9 + 2(-3) = 7$ — steadier than the
   $13$ independence would predict. That makes sense: negative covariance
   means the two tend to miss in opposite directions, so their deviations
   partly cancel in the sum.

</details>

<details><summary><b>Checkpoint 7</b></summary>

1. Scale law: $\operatorname{Var}[2X] = 4 \cdot \tfrac14 = 1$.
   Independent-sum law: $\operatorname{Var}[2X + Y] = 1 + 1 = 2 =
   \tfrac21$, already in lowest terms with $q > 0$, so
   $p + q = 2 + 1 = 3$. **Answer (B).** Seeded check
   ($\sigma = 0.5$ and $1$, 300,000 runs): variance estimate 2.0114.

2. Every rational number has exactly one representation with
   $\gcd(p, q) = 1$ and $q > 0$, so exactly one option can be correct. It
   catches the classic error of submitting an unreduced fraction (e.g.
   $\tfrac{70}{12}$ for $\tfrac{35}{6}$) as if it were a different answer.

</details>

<details><summary><b>Checkpoint 8</b></summary>

1. First classmate: small-sample bias — 8-of-10 heads happens about 5.5%
   of the time with a perfectly fair coin, so $n = 10$ justifies no
   conclusion. Second classmate: the gambler's fallacy — independent flips
   have no memory, so nothing is ever "due".

2. $X$ uniform on $\{-1, 0, 1\}$ and $Y = X^2$: the covariance is exactly
   0 even though $Y$ is a function of $X$. Covariance only sees
   straight-line co-movement, and this dependence is perfectly symmetric —
   positive and negative $x$ deviations pair with the *same* $y$ values,
   so the deviation products cancel exactly.

</details>

<details><summary><b>Checkpoint 9</b></summary>

1. $\operatorname{Var}[S] = c^2 \cdot n \sigma^2 = c^2 \cdot 400 \cdot 2
   = 1$, so $c = \tfrac{1}{\sqrt{800}} = \tfrac{1}{20\sqrt2}
   \approx 0.0354$.

2. `study` and `score` — their large positive covariance (Section 6) says
   they carry overlapping information, so a compression method could
   summarize the pair with one direction. `shoe`'s near-zero covariance
   with both marks it as an independent direction of its own.

</details>